# Court Document Pipeline (Colab GPU)

Downloads court PDFs and extracts text — all on Google's servers using a free GPU.
No files need to leave your laptop. No NVIDIA API quota consumed.

### What this notebook does
1. **Clones the repo** — gets the scraper code and `valid_cases.json` (the list of known case numbers)
2. **Downloads PDFs** from the court API directly into Colab
3. **Extracts text** — PyMuPDF first (free, instant), then Qwen2-VL-7B on GPU for scanned pages
4. **Saves to Google Drive** (persistent) and/or lets you download a zip

### Team workflow
All team members should use the **same shared Google Drive folder**. The notebook automatically
skips any case whose PDFs are already in the folder, so no duplicates.

To split the work, each person sets a different `MY_WORKER` / `TOTAL_WORKERS` in the config cell
(e.g. person 1 = `0/3`, person 2 = `1/3`, person 3 = `2/3`). Each person gets a unique slice of cases.

### Before you start
- **Runtime → Change runtime type → T4 GPU**
- Have a browser tab open to get a court session ID when prompted
- If working as a team: one person creates a shared Drive folder and shares it with the others

In [1]:
# !pip install -q pymupdf transformers>=4.45 accelerate bitsandbytes qwen-vl-utils Pillow

# import torch

# if not torch.cuda.is_available():
#     raise RuntimeError(
#         "No GPU detected!\n"
#         "Go to Runtime > Change runtime type > T4 GPU"
#     )

# gpu_name = torch.cuda.get_device_name(0)
# gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
# print(f"GPU ready: {gpu_name} ({gpu_mem:.1f} GB)")

## Setup

Clone the repo (gets the scraper code + `valid_cases.json`) and optionally
mount Google Drive so downloaded files persist between sessions.

In [2]:
import os
from pathlib import Path

# ============================================================
# CONFIGURATION — edit these if needed
# ============================================================
REPO_URL = "https://github.com/karimiborna/litigation-outcome-pipeline.git"
BRANCH   = "phase-1/scraper-and-data"

USE_DRIVE = True   # Set False to skip Drive (files won't persist)
DRIVE_DIR = "/content/drive/MyDrive/litigation-pipeline"

# TEAM WORK SPLITTING — divide cases across team members
# Each person picks a unique MY_WORKER number (0-indexed).
# Example for 3 people: person A = 0/3, person B = 1/3, person C = 2/3
# Set TOTAL_WORKERS = 1 to download everything yourself.
MY_WORKER      = 0   # Your worker number (0, 1, 2, ...)
TOTAL_WORKERS  = 1   # Total number of team members downloading
# ============================================================

REPO_DIR = "/content/litigation-outcome-pipeline"

if not Path(REPO_DIR).exists():
    !git config --global credential.helper ''
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Install the project so we can import scraper modules
!pip install -q -e {REPO_DIR}

# Set up output directories
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PDF_DIR    = f"{DRIVE_DIR}/pdfs"
    OUTPUT_DIR = f"{DRIVE_DIR}/extracted"
else:
    PDF_DIR    = f"{REPO_DIR}/data/raw/pdfs"
    OUTPUT_DIR = f"{REPO_DIR}/data/processed/extracted"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nPDFs will be saved to:  {PDF_DIR}")
print(f"Text will be saved to: {OUTPUT_DIR}")

Repo already cloned at /content/litigation-outcome-pipeline
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for litigation-outcome-pipeline (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PDFs will be saved to:  /content/drive/MyDrive/litigation-pipeline/pdfs
Text will be saved to: /content/drive/MyDrive/litigation-pipeline/extracted


## Step 1 — Download PDFs from the court API

Reads `valid_cases.json` from the cloned repo and downloads documents for each case.
**Only useful document types are downloaded** (claims, judgments, orders, dismissals).
Procedural docs (proof of service, continuances, etc.) are skipped automatically.

**To get a session ID:**
1. Open https://webapps.sftc.org/cc/CaseCalendar.dll in your browser
2. Copy the hex string after `SessionID=` in the URL

In [3]:
import json
import re
import time

import requests

BASE_URL  = "https://webapps.sftc.org"
CASE_PATH = "/ci/CaseInfo.dll"
USER_AGENT = "MSDS603-Research-Scraper/1.0 (SF Small Claims academic study)"
DOWNLOAD_DELAY = 2.5  # seconds between requests


def ask_session_id():
    print("\n" + "=" * 50)
    print("SESSION ID NEEDED")
    print("1. Open https://webapps.sftc.org/cc/CaseCalendar.dll")
    print("2. Copy the hex string after SessionID= from the URL")
    print("=" * 50)
    sid = input("Paste SessionID: ").strip()
    return sid


def get_documents(case_num, session_id):
    url = (
        f"{BASE_URL}{CASE_PATH}"
        f"/datasnap/rest/TServerMethods1/GetDocuments"
        f"/{case_num}/{session_id}/"
    )
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if data["result"][0] == -1:
        raise Exception("SESSION_EXPIRED")
    if data["result"][0] == 0:
        return []
    return json.loads(data["result"][1])


def sanitize(desc, max_len=40):
    return re.sub(r"[^\w\-]", "_", desc)[:max_len]


DOC_TYPE_WHITELIST = {
    "CLAIM_OF_PLAINTIFF", "DEFENDANT_S_CLAIM", "JUDGMENT", "ORDER",
    "DISMISSAL", "Notice_of_Entry_of_Judgment", "DECLARATION_OF_APPEARANCE",
    "STIPULATION", "COURT_JUDGMENT",
}


def is_doc_wanted(description):
    desc_upper = description.upper()
    return any(w.upper() in desc_upper for w in DOC_TYPE_WHITELIST)


# Load valid case numbers from the cloned repo
valid_cases_path = Path(REPO_DIR) / "scraper" / "state" / "valid_cases.json"
with open(valid_cases_path) as f:
    store = json.load(f)
all_valid = sorted(store.get("valid", {}).keys())
print(f"Valid cases from repo: {len(all_valid)}")

# Split across workers — each person gets every Nth case
my_cases = [c for i, c in enumerate(all_valid) if i % TOTAL_WORKERS == MY_WORKER]
print(f"Worker {MY_WORKER}/{TOTAL_WORKERS}: assigned {len(my_cases)} cases")

# Skip cases whose PDFs are already in the shared Drive folder
to_download = []
for case_num in my_cases:
    existing = list(Path(PDF_DIR).glob(f"{case_num}_*.pdf"))
    if not existing:
        to_download.append(case_num)

print(f"Already in Drive (skipped): {len(my_cases) - len(to_download)}")
print(f"To download: {len(to_download)}")

if to_download:
    session_id = ask_session_id()
    http = requests.Session()
    http.headers["User-Agent"] = USER_AGENT
    total_pdfs = 0

    for i, case_num in enumerate(to_download):
        # Re-check in case a teammate downloaded this while we were running
        if list(Path(PDF_DIR).glob(f"{case_num}_*.pdf")):
            continue

        try:
            time.sleep(DOWNLOAD_DELAY)
            docs = get_documents(case_num, session_id)
        except Exception as e:
            if "SESSION_EXPIRED" in str(e):
                session_id = ask_session_id()
                time.sleep(DOWNLOAD_DELAY)
                docs = get_documents(case_num, session_id)
            else:
                print(f"  [{i + 1}/{len(to_download)}] {case_num} — ERROR: {e}")
                continue

        case_pdfs = 0
        for doc in docs:
            doc_url = doc.get("URL", "")
            desc = doc.get("DESCRIPTION", "doc")
            if not doc_url or not is_doc_wanted(desc):
                continue

            pdf_path = Path(PDF_DIR) / f"{case_num}_{sanitize(desc)}.pdf"
            if pdf_path.exists():
                continue

            time.sleep(DOWNLOAD_DELAY)
            try:
                resp = http.get(doc_url, timeout=60, stream=True)
                if "pdf" in resp.headers.get("content-type", "").lower():
                    with open(pdf_path, "wb") as f:
                        for chunk in resp.iter_content(8192):
                            f.write(chunk)
                    total_pdfs += 1
                    case_pdfs += 1
            except Exception as e:
                print(f"  Download failed: {e}")

        print(f"  [{i + 1}/{len(to_download)}] {case_num} — {case_pdfs} PDFs downloaded (total: {total_pdfs})")

    print(f"\nDownloaded {total_pdfs} PDFs")
else:
    print("All PDFs already downloaded.")

# List what we have
all_pdfs = sorted(f.name for f in Path(PDF_DIR).glob("*.pdf"))
print(f"\nTotal PDFs on disk: {len(all_pdfs)}")

Valid cases from repo: 800
Worker 0/1: assigned 800 cases
Already in Drive (skipped): 800
To download: 0
All PDFs already downloaded.

Total PDFs on disk: 2367


## Step 2 — PyMuPDF (free text extraction)

Pulls selectable text from each PDF instantly. No GPU needed.
Only PDFs that are scanned images (< 100 chars of text) move to the GPU step.

In [4]:
import fitz  # PyMuPDF

MIN_TEXT_LENGTH = 100


def extract_with_pymupdf(pdf_path):
    doc = fitz.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages).strip()


all_pdfs = sorted(Path(PDF_DIR).glob("*.pdf"))

# Only extract whitelisted document types — skip procedural PDFs
pdfs = [p for p in all_pdfs if is_doc_wanted(p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem)]
skipped_types = len(all_pdfs) - len(pdfs)

already_done = []
pymupdf_extracted = []
needs_gpu = []
corrupted = []

for pdf_path in pdfs:
    
    txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

    if txt_path.exists():
        already_done.append(pdf_path)
        continue

    print(f"Processing: {pdf_path.name}")
    try:
        text = extract_with_pymupdf(pdf_path)
    except Exception as e:
        print(f"  CORRUPTED: {pdf_path.name} — {e}")
        corrupted.append(pdf_path)
        continue

    if len(text) > MIN_TEXT_LENGTH:
        txt_path.write_text(text, encoding="utf-8")
        pymupdf_extracted.append(pdf_path)
    else:
        needs_gpu.append(pdf_path)

print(f"Skipped (not in whitelist):  {skipped_types}")
print(f"Already extracted (skipped): {len(already_done)}")
for name in [p.name for p in already_done]:
    print(f"  {name}")
print(f"PyMuPDF extracted this run:  {len(pymupdf_extracted)}")
for name in [p.name for p in pymupdf_extracted]:
    print(f"  {name}")
print(f"Need GPU vision model:       {len(needs_gpu)}")
for name in [p.name for p in needs_gpu]:
    print(f"  {name}")
print(f"Corrupted (skipped):         {len(corrupted)}")
for name in [p.name for p in corrupted]:
    print(f"  {name}")

Processing: CSM25870166_Notice_of_Entry_of_Judgment.pdf
  CORRUPTED: CSM25870166_Notice_of_Entry_of_Judgment.pdf — Failed to open file '/content/drive/MyDrive/litigation-pipeline/pdfs/CSM25870166_Notice_of_Entry_of_Judgment.pdf'.
Processing: CSM25870394_REQUEST_TO_CORRECT_OR_CANCEL_JUDGMENT_AN.pdf
Processing: CSM25870394_SATISFACTION_OF_JUDGMENT.pdf
Processing: CSM25870395_JUDGMENT_ON_PLAINTIFF_S_CLAIM.pdf
Processing: CSM25870396_JUDGMENT_ON_PLAINTIFF_S_CLAIM.pdf
Processing: CSM25870397_ORDER.pdf
Processing: CSM25870398_DISMISSAL.pdf
Processing: CSM25870398_JUDGMENT_ON_PLAINTIFF_S_CLAIM.pdf
Processing: CSM25870398_ORDER.pdf
Processing: CSM25870399_DISMISSAL_OF_ENTIRE_ACTION.pdf
Processing: CSM25870399_ORDER.pdf
Processing: CSM25870400_DISMISSAL_OF_ENTIRE_ACTION.pdf
Processing: CSM25870400_ORDER.pdf
Processing: CSM25870401_JUDGMENT_ON_PLAINTIFF_S_CLAIM.pdf
Processing: CSM25870401_ORDER.pdf
Processing: CSM25870402_JUDGMENT_ON_PLAINTIFF_S_CLAIM.pdf
Processing: CSM25870402_ORDER.pdf
Proces

In [5]:
from collections import Counter

doc_types = []
for p in pdfs:
    parts = p.stem.split("_", 1)
    doc_type = parts[1] if len(parts) > 1 else p.stem
    doc_types.append(doc_type)

counts = Counter(doc_types)
print(f"Total whitelisted PDFs: {len(pdfs)}\n")
for doc_type, count in counts.most_common():
    print(f"  {count:>5}  {doc_type}")


Total whitelisted PDFs: 2254

    479  ORDER
    427  JUDGMENT_ON_PLAINTIFF_S_CLAIM
    421  Notice_of_Entry_of_Judgment
    242  Small_Claims_-_Continue_Order
    203  Small_Claims_Order_of_Dismissal_of_Entir
    114  DISMISSAL_OF_ENTIRE_ACTION
     65  STIPULATION
     63  CLAIM_OF_PLAINTIFF
     62  SATISFACTION_OF_JUDGMENT
     25  COURT_JUDGMENT_SMALL_CLAIMS_APPEAL
     18  REQUEST_TO_CORRECT_OR_CANCEL_JUDGMENT_AN
     15  ORDER_OF_EXAMINATION_ISSUED
     14  DECLARATION_OF_APPEARANCE_ON_BEHALF_OF_C
      9  ABSTRACT_OF_JUDGMENT_FILED_BY_PLAINTIFF
      9  MOTION_TO_VACATE_JUDGMENT_FILED_BY_DEFEN
      8  JUDGMENT_CREDITOR_REQUEST_FOR_FUNDS_FILE
      7  ORDER_OF_EXAMINATION_RETURNED_SERVED__SE
      7  Order_after_Hearing_and_Notice_of_Time_a
      6  Discharge_Order
      5  DISMISSAL
      5  ORDER_ON_COURT_FEE_WAIVER
      5  Order_to_Dismiss_Small_Claims_Appeal
      4  Small_Claims_Motion_-_Continue_Order
      4  DEPOSIT_OF_JUDGMENT_DEBTOR__TRAN___A5626
      3  Off_Calenda

## Step 3 — Load Vision Model

Loads **Qwen2-VL-7B-Instruct** with 4-bit quantization (~4 GB VRAM).
First run downloads ~4 GB of weights (cached for subsequent runs).

Skipped entirely if all PDFs had selectable text.

In [6]:

# import os
# os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

# model = None
# processor = None

# if needs_gpu:
#     from transformers import (
#         AutoProcessor,
#         BitsAndBytesConfig,
#         Qwen2VLForConditionalGeneration,
#     )
#     from huggingface_hub import snapshot_download

#     MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

#     print(f"Step 1/4: Downloading model files for {MODEL_ID}...")
#     model_path = snapshot_download(MODEL_ID)
#     print(f"  Download complete: {model_path}")

#     print("Step 2/4: Loading model into GPU (4-bit quantized)...")
#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_compute_dtype=torch.bfloat16,
#     )
#     model = Qwen2VLForConditionalGeneration.from_pretrained(
#         model_path,
#         quantization_config=bnb_config,
#         device_map="auto",
#         torch_dtype=torch.bfloat16,
#     )
#     print("  Model loaded.")

#     print("Step 3/4: Loading processor/tokenizer...")
#     processor = AutoProcessor.from_pretrained(model_path)
#     print("  Processor loaded.")

#     used = torch.cuda.memory_allocated() / 1e9
#     print(f"Step 4/4: Ready. GPU memory used: {used:.1f} GB")
# else:
#     print("All PDFs had selectable text — no model needed.")


## Step 4 — Extract scanned PDFs with GPU

Renders each page to an image and sends it through the vision model.
Uses a **targeted prompt** for claim documents (SC-100 forms) that skips bilingual
boilerplate and focuses on the filled-in case data. Other document types get the
generic extraction prompt. Pages with no case data are skipped entirely.

In [7]:
# import io

# from PIL import Image
# from qwen_vl_utils import process_vision_info

# GENERIC_PROMPT = (
#     "This is a page from a San Francisco small claims court document. "
#     "Extract all text exactly as it appears. Include party names, dates, "
#     "claim amounts, rulings, and any other information on the page. "
#     "Output plain text only."
# )

# CLAIM_PROMPT = (
#     "This is a page from a California SC-100 small claims complaint form. "
#     "SKIP all boilerplate instructions, bilingual notices, and form guidance. "
#     "Extract ONLY the filled-in case-specific information:\n"
#     "- Plaintiff name(s), address, phone\n"
#     "- Defendant name(s), address, phone\n"
#     "- Trial date and department\n"
#     "- Dollar amount claimed and basis for the amount\n"
#     "- Description of what happened (the plaintiff's narrative)\n"
#     "- Any evidence, witnesses, or documents referenced\n"
#     "- Whether plaintiff tried to resolve before suing\n\n"
#     "If a page is only boilerplate instructions with no filled-in data, "
#     "respond with: [NO CASE DATA ON THIS PAGE]\n"
#     "Output plain text only."
# )
# DPI = 150


# def get_prompt_for_doc(filename):
#     name_upper = filename.upper()
#     if "CLAIM_OF_PLAINTIFF" in name_upper or "DEFENDANT_S_CLAIM" in name_upper:
#         return CLAIM_PROMPT
#     return GENERIC_PROMPT


# def page_to_pil(page, dpi=DPI):
#     mat = fitz.Matrix(dpi / 72, dpi / 72)
#     pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
#     return Image.open(io.BytesIO(pix.tobytes("png")))


# def extract_page(pil_image, prompt):
#     messages = [
#         {
#             "role": "user",
#             "content": [
#                 {"type": "image", "image": pil_image},
#                 {"type": "text", "text": prompt},
#             ],
#         }
#     ]

#     text_input = processor.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True
#     )
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs = processor(
#         text=[text_input],
#         images=image_inputs,
#         videos=video_inputs,
#         padding=True,
#         return_tensors="pt",
#     ).to("cuda")

#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs, max_new_tokens=2048, do_sample=False
#         )

#     generated = output_ids[0, inputs.input_ids.shape[1] :]
#     return processor.decode(generated, skip_special_tokens=True)


# if needs_gpu and model is not None:
#     t0 = time.time()
#     skipped = 0
#     extracted = 0
#     total_pages = 0
#     skipped_pages = 0

#     print(f"Starting GPU extraction for {len(needs_gpu)} scanned PDFs...\n")

#     for i, pdf_path in enumerate(needs_gpu):
#         txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

#         if txt_path.exists():
#             skipped += 1
#             print(f"  [{i + 1}/{len(needs_gpu)}] SKIPPED (already extracted): {pdf_path.name}")
#             continue

#         prompt_type = "CLAIM" if "CLAIM" in pdf_path.name.upper() else "GENERIC"
#         prompt = get_prompt_for_doc(pdf_path.name)
#         doc = fitz.open(str(pdf_path))
#         pages_text = []

#         print(f"  [{i + 1}/{len(needs_gpu)}] {pdf_path.name} ({len(doc)} pages, prompt: {prompt_type})")

#         pdf_t0 = time.time()
#         for page_num, page in enumerate(doc, start=1):
#             page_t0 = time.time()
#             img = page_to_pil(page)
#             text = extract_page(img, prompt)
#             page_elapsed = time.time() - page_t0
#             total_pages += 1

#             if "[NO CASE DATA ON THIS PAGE]" not in text:
#                 pages_text.append(text)
#                 print(f"    page {page_num}/{len(doc)} — {len(text):,} chars extracted ({page_elapsed:.1f}s)")
#             else:
#                 skipped_pages += 1
#                 print(f"    page {page_num}/{len(doc)} — no case data, skipped ({page_elapsed:.1f}s)")

#         doc.close()
#         torch.cuda.empty_cache()

#         full_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)
#         txt_path.write_text(full_text, encoding="utf-8")
#         extracted += 1
#         pdf_elapsed = time.time() - pdf_t0
#         print(f"    -> Saved: {txt_path.name} ({len(full_text):,} chars, {len(pages_text)} pages kept, {pdf_elapsed:.1f}s)")

#         # Progress summary every 10 PDFs
#         if (i + 1) % 10 == 0:
#             elapsed = time.time() - t0
#             rate = (extracted + skipped) / elapsed * 60
#             remaining = len(needs_gpu) - (i + 1)
#             eta_min = remaining / rate if rate > 0 else 0
#             print(f"\n  --- Progress: {i + 1}/{len(needs_gpu)} | "
#                   f"Extracted: {extracted} | Skipped: {skipped} | "
#                   f"Rate: {rate:.1f} PDFs/min | ETA: {eta_min:.0f} min ---\n")

#     elapsed = time.time() - t0
#     print(f"\n{'=' * 50}")
#     print(f"GPU extraction complete in {elapsed:.0f}s ({elapsed / 60:.1f} min)")
#     print(f"  PDFs extracted:    {extracted}")
#     print(f"  PDFs skipped:      {skipped}")
#     print(f"  Total pages seen:  {total_pages}")
#     print(f"  Pages with data:   {total_pages - skipped_pages}")
#     print(f"  Pages skipped:     {skipped_pages} (no case data)")
#     if extracted > 0:
#         print(f"  Avg time per PDF:  {elapsed / extracted:.1f}s")
#     print(f"{'=' * 50}")
# else:
#     print("No scanned PDFs to process.")

## Step 4b — NVIDIA API fallback (no GPU needed)

Picks up where the GPU extraction left off. Sends remaining scanned PDFs to
**NVIDIA's Llama 3.2 90B Vision** API — runs on NVIDIA's servers, no local GPU required.

Uses the same prompts (CLAIM vs GENERIC) and page-skipping logic as the GPU cell.
Output goes to a separate `extracted_nvidia/` folder to keep them organized.

**Limits:** 500 API calls/day per key. Each team member can run their own key in parallel.
Get a free key at https://build.nvidia.com

In [ ]:
import base64
import io
import os

from openai import OpenAI

# ============================================================
# NVIDIA API CONFIG
# ============================================================
NVIDIA_API_KEY = input("NVIDIA API key (leave blank to skip): ").strip()
NVIDIA_API_BASE = "https://integrate.api.nvidia.com/v1"
NVIDIA_MODEL = "meta/llama-3.2-90b-vision-instruct"
NVIDIA_DAILY_CAP = 490  # stay under the 500/day limit

# Output goes to a separate folder to keep GPU and API extractions organized
NVIDIA_OUTPUT_DIR = OUTPUT_DIR.rstrip("/").rsplit("/", 1)[0] + "/extracted_nvidia"
os.makedirs(NVIDIA_OUTPUT_DIR, exist_ok=True)
print(f"NVIDIA extractions will be saved to: {NVIDIA_OUTPUT_DIR}")

# Same prompts as the GPU cell
GENERIC_PROMPT_NV = (
    "This is a page from a San Francisco small claims court document. "
    "Extract all text exactly as it appears. Include party names, dates, "
    "claim amounts, rulings, and any other information on the page. "
    "Output plain text only."
)

CLAIM_PROMPT_NV = (
    "This is a page from a California SC-100 small claims complaint form. "
    "SKIP all boilerplate instructions, bilingual notices, and form guidance. "
    "Extract ONLY the filled-in case-specific information:\n"
    "- Plaintiff name(s), address, phone\n"
    "- Defendant name(s), address, phone\n"
    "- Trial date and department\n"
    "- Dollar amount claimed and basis for the amount\n"
    "- Description of what happened (the plaintiff's narrative)\n"
    "- Any evidence, witnesses, or documents referenced\n"
    "- Whether plaintiff tried to resolve before suing\n\n"
    "If a page is only boilerplate instructions with no filled-in data, "
    "respond with: [NO CASE DATA ON THIS PAGE]\n"
    "Output plain text only."
)


def get_prompt_for_doc_nv(filename):
    name_upper = filename.upper()
    if "CLAIM_OF_PLAINTIFF" in name_upper or "DEFENDANT_S_CLAIM" in name_upper:
        return CLAIM_PROMPT_NV
    return GENERIC_PROMPT_NV


def page_to_base64(page, dpi=150):
    """Render a PDF page to JPEG and return as base64 string."""
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return base64.b64encode(pix.tobytes("jpeg")).decode("utf-8")


if NVIDIA_API_KEY:
    client = OpenAI(base_url=NVIDIA_API_BASE, api_key=NVIDIA_API_KEY)

    # Find PDFs that still need extraction:
    # not in OUTPUT_DIR (GPU extractions) AND not in NVIDIA_OUTPUT_DIR
    remaining = []
    for pdf_path in needs_gpu:
        gpu_txt = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"
        nv_txt = Path(NVIDIA_OUTPUT_DIR) / f"{pdf_path.stem}.txt"
        if not gpu_txt.exists() and not nv_txt.exists():
            remaining.append(pdf_path)

    print(f"\nTotal scanned PDFs (needs_gpu):  {len(needs_gpu)}")
    print(f"Already done (GPU):              {len(needs_gpu) - len(remaining) - len([p for p in needs_gpu if Path(NVIDIA_OUTPUT_DIR, p.stem + '.txt').exists()])}")
    print(f"Already done (NVIDIA):           {len([p for p in needs_gpu if Path(NVIDIA_OUTPUT_DIR, p.stem + '.txt').exists()])}")
    print(f"Remaining to extract:            {len(remaining)}")
    print(f"Daily cap:                       {NVIDIA_DAILY_CAP} API calls\n")

    t0 = time.time()
    extracted = 0
    total_pages = 0
    skipped_pages = 0
    api_calls = 0

    for i, pdf_path in enumerate(remaining):
        if api_calls >= NVIDIA_DAILY_CAP:
            print(f"\n  Daily cap reached ({NVIDIA_DAILY_CAP} API calls). "
                  f"Run again tomorrow or use a different API key.")
            break

        prompt_type = "CLAIM" if "CLAIM" in pdf_path.name.upper() else "GENERIC"
        prompt = get_prompt_for_doc_nv(pdf_path.name)
        doc = fitz.open(str(pdf_path))
        pages_text = []

        print(f"  [{i + 1}/{len(remaining)}] {pdf_path.name} ({len(doc)} pages, prompt: {prompt_type})")

        pdf_t0 = time.time()
        for page_num, page in enumerate(doc, start=1):
            if api_calls >= NVIDIA_DAILY_CAP:
                print(f"    Daily cap reached mid-PDF — saving what we have")
                break

            page_t0 = time.time()
            img_b64 = page_to_base64(page)

            try:
                response = client.chat.completions.create(
                    model=NVIDIA_MODEL,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": f"data:image/jpeg;base64,{img_b64}"
                                    },
                                },
                            ],
                        }
                    ],
                    max_tokens=2048,
                    temperature=0,
                )
                text = response.choices[0].message.content or ""
                api_calls += 1
            except Exception as e:
                print(f"    page {page_num}/{len(doc)} — API ERROR: {e}")
                continue

            page_elapsed = time.time() - page_t0
            total_pages += 1

            if "[NO CASE DATA ON THIS PAGE]" not in text:
                pages_text.append(text)
                print(f"    page {page_num}/{len(doc)} — {len(text):,} chars extracted ({page_elapsed:.1f}s) [API call {api_calls}/{NVIDIA_DAILY_CAP}]")
            else:
                skipped_pages += 1
                print(f"    page {page_num}/{len(doc)} — no case data, skipped ({page_elapsed:.1f}s) [API call {api_calls}/{NVIDIA_DAILY_CAP}]")

        doc.close()

        if pages_text:
            full_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)
            txt_path = Path(NVIDIA_OUTPUT_DIR) / f"{pdf_path.stem}.txt"
            txt_path.write_text(full_text, encoding="utf-8")
            extracted += 1
            pdf_elapsed = time.time() - pdf_t0
            print(f"    -> Saved: {txt_path.name} ({len(full_text):,} chars, {len(pages_text)} pages kept, {pdf_elapsed:.1f}s)")
        else:
            print(f"    -> No text extracted, skipping save")

        # Progress summary every 10 PDFs
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            rate = extracted / elapsed * 60 if elapsed > 0 else 0
            left = len(remaining) - (i + 1)
            eta_min = left / rate if rate > 0 else 0
            print(f"\n  --- Progress: {i + 1}/{len(remaining)} | "
                  f"Extracted: {extracted} | API calls: {api_calls}/{NVIDIA_DAILY_CAP} | "
                  f"Rate: {rate:.1f} PDFs/min | ETA: {eta_min:.0f} min ---\n")

    elapsed = time.time() - t0
    print(f"\n{'=' * 50}")
    print(f"NVIDIA API extraction complete in {elapsed:.0f}s ({elapsed / 60:.1f} min)")
    print(f"  PDFs extracted:    {extracted}")
    print(f"  API calls used:    {api_calls}/{NVIDIA_DAILY_CAP}")
    print(f"  Total pages seen:  {total_pages}")
    print(f"  Pages with data:   {total_pages - skipped_pages}")
    print(f"  Pages skipped:     {skipped_pages} (no case data)")
    if extracted > 0:
        print(f"  Avg time per PDF:  {elapsed / extracted:.1f}s")
    still_remaining = len(remaining) - extracted
    if still_remaining > 0:
        print(f"  Still remaining:   {still_remaining} PDFs (run again with fresh key/day)")
    print(f"{'=' * 50}")
else:
    print("Skipping NVIDIA extraction (no API key).")

NVIDIA extractions will be saved to: /content/drive/MyDrive/litigation-pipeline/extracted_nvidia

Total scanned PDFs (needs_gpu):  656
Already done (GPU):              0
Already done (NVIDIA):           492
Remaining to extract:            164
Daily cap:                       490 API calls

  [1/164] CSM25870695_DISMISSAL_OF_ENTIRE_ACTION.pdf (2 pages, prompt: GENERIC)
    page 1/2 — 651 chars extracted (10.5s) [API call 1/490]
    page 2/2 — 1,399 chars extracted (18.5s) [API call 2/490]
    -> Saved: CSM25870695_DISMISSAL_OF_ENTIRE_ACTION.txt (2,072 chars, 2 pages kept, 29.0s)
  [2/164] CSM25870696_ORDER.pdf (1 pages, prompt: GENERIC)
    page 1/1 — 1,016 chars extracted (15.4s) [API call 3/490]
    -> Saved: CSM25870696_ORDER.txt (1,016 chars, 1 pages kept, 15.4s)
  [3/164] CSM25870697_ORDER.pdf (1 pages, prompt: GENERIC)
    page 1/1 — 1,173 chars extracted (17.2s) [API call 4/490]
    -> Saved: CSM25870697_ORDER.txt (1,173 chars, 1 pages kept, 17.2s)
  [4/164] CSM25870698_JUDGMENT

## Step 5 — Extract labels with GPT-4o-mini

Sends outcome documents (judgments, orders, dismissals) to GPT-4o-mini to extract
structured labels: win/loss/dismissed, amount awarded, whether defendant appeared, etc.

Requires an OpenAI API key (set `LLM_API_KEY` below). Costs ~$0.001 per case.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from openai import OpenAI

LLM_API_KEY = input("OpenAI API key (leave blank to skip label extraction): ").strip()

if LLM_API_KEY:
    from features.labels import LabelExtractor, CaseLabels, gather_outcome_text
    from features.config import FeaturesConfig

    label_config = FeaturesConfig(
        llm_api_key=LLM_API_KEY,
        llm_model="gpt-4o-mini",
        cache_dir=f"{OUTPUT_DIR}/../labels_cache",
    )
    extractor = LabelExtractor(config=label_config)

    txt_dir = Path(OUTPUT_DIR)
    case_numbers = sorted({
        f.name.split("_")[0]
        for f in txt_dir.glob("*.txt")
        if f.name.startswith("CSM")
    })
    print(f"Found {len(case_numbers)} cases to label")

    all_labels = {}
    for i, case_num in enumerate(case_numbers):
        outcome_text = gather_outcome_text(case_num, txt_dir)
        if not outcome_text:
            continue
        labels = extractor.extract_labels(case_num, txt_dir)
        if labels:
            all_labels[case_num] = labels.model_dump()
            print(f"  [{i+1}/{len(case_numbers)}] {case_num}: {labels.outcome} "
                  f"(${labels.total_awarded or 0:.2f})")

    labels_path = Path(OUTPUT_DIR) / "../labels.json"
    labels_path.write_text(json.dumps(all_labels, indent=2), encoding="utf-8")
    print(f"\nLabels saved: {labels_path} ({len(all_labels)} cases)")
else:
    print("Skipping label extraction (no API key).")

## Results

Summary of all extracted text and labels. Downloads everything as a zip.

In [ ]:
import shutil

txt_files = sorted(Path(OUTPUT_DIR).glob("*.txt"))
total_chars = sum(f.stat().st_size for f in txt_files)

print(f"Total text files: {len(txt_files)}")
print(f"Total size:       {total_chars:,} characters ({total_chars / 1024:.0f} KB)")
print()

for f in txt_files[:20]:
    print(f"  {f.name}  ({f.stat().st_size:,} chars)")
if len(txt_files) > 20:
    print(f"  ... and {len(txt_files) - 20} more")

labels_path = Path(OUTPUT_DIR) / "../labels.json"
if labels_path.exists():
    labels = json.loads(labels_path.read_text())
    outcomes = {}
    for v in labels.values():
        o = v.get("outcome", "unknown") or "unknown"
        outcomes[o] = outcomes.get(o, 0) + 1
    print(f"\nLabels: {len(labels)} cases")
    for outcome, count in sorted(outcomes.items()):
        print(f"  {outcome}: {count}")

if USE_DRIVE:
    print(f"\nFiles are saved to Google Drive at: {DRIVE_DIR}")
    print("They will persist even if this runtime disconnects.")

zip_path = "/content/extracted_texts"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"\nZip: {zip_path}.zip ({Path(zip_path + '.zip').stat().st_size / 1024:.0f} KB)")

from google.colab import files
files.download(f"{zip_path}.zip")